## Preliminary lab: Building and Training a Neural Network on the Iris Dataset
In this lab, you will build a simple neural network using PyTorch to classify the Iris dataset into three categories. The neural network will consist of:  
- Input Layer: 4 features (from the Iris dataset)
- Hidden Layer: 4 neurons, reduced to 2 neurons using ReLU activation
- Output Layer: 3 neurons (for the three classes), using Softmax activation

You will train the network using Gradient Descent for a few steps and evaluate its performance. Additionally, you will explore the “latent space” (the intermediate space after the first hidden layer).

#### 2. Load the Iris Dataset
The Iris dataset is available from scikit-learn.
It contains 150 samples of iris flowers, each sample having 4 features (sepal length, sepal width, petal length, petal width), and each sample is labeled with one of three species: Setosa, Versicolor, or Virginica.

- Use sklearn.datasets.load_iris() to load the dataset.
- Split the dataset into features (X) and targets (y).
- Encode the target labels into integers (0, 1, 2) since you will be using CrossEntropyLoss which expects integer labels.

In [ ]:
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

In [ ]:
x,y = load_iris(return_X_y=True, as_frame=True)

# data
display(x)

# labels
display(y)

# Some data inspection to get used to it

- What is the target? Name of the target column?  
- Data type of the target: continuous = regression OR categorical = classification?  
- Is it supervised or unsupervised learning?  
- Inspect number of rows and columns in train and test set. Do they have the same features?  

The target is a Pandas Series of 150 values.  
The target is not a continuous value, it can have value 0,1 or 2 --> these are classes --> (supervised) classification problem.  
The data has 150 rows and 4 columns, while the corresponding labels have 150 rows.  

In [ ]:
print(x.shape)

print()

print(y.shape)

In [ ]:
print('DESCRIBE:')
print(x.describe())
print()
print(y.describe())
print()

print('INFO')
print(x.info())
print()
print(y.info())

In [ ]:
y.value_counts()

- Inspect missing values PER COLUMN:
    - compute missing value % per feature:
        - missing_rate < 0.05 → keep, simple impute
        - 0.05 ≤ missing_rate ≤ 0.3 → keep, better impute --> (SimpleImputer) / KNNImputer --> **FOR NUMERIC COLUMNS**
        - missing_rate > 0.5 → usually drop unless clearly important

No missing values at all.

In [ ]:
# Nan values per column
print(f"Nan values per column: \n{x.isna().sum()}")
print()

# Nan values per row
print(f"Nan values per row: \n{x.isna().sum(axis=1)}")
print()

# Nan values in the whole DF
print(f"All Nan values: {x.isna().sum().sum()}")

Identify all data types:  
data --> all columns are float64  
target --> all labels are int64  

In [ ]:
print(x.info())
print()
print(y.info())

#### Plot to visualize the distribution of the data (+ to practice)  
Are there any anomalies?
- **Outliers**: Look for bars that are far away from the rest of the data. These are extreme values that could be errors or rare cases.
- **Skewness**: If the histogram has a long tail on one side, the data might be skewed (right or left). This might need transformation.
- **Bimodal/Multimodal**: Two or more peaks in the histogram could suggest multiple groups in the data (e.g., different species).
- **Uniform Distribution**: If all values are spread evenly, there might be an issue with data collection or the feature might not be informative.
- **Normal Distribution**: A bell-shaped curve is ideal for many models, but if the data doesn’t look normal, you may need to transform it.
- **Gaps**: Large spaces with no data might indicate missing or underrepresented values.
- **Overlapping Data**: Look for features that separate the classes well or those that overlap, which could make classification harder.

In [ ]:
plt.figure(figsize=(10,10))

for i, col in enumerate(x):
    subplot_index = i + 1
    plt.subplot(5,4,subplot_index)      # number of rows (if I put 1 it gets too stretched), number of columns, index of where you're plotting right now
    sns.histplot(x[col], kde = True)
    plt.title(col)
plt.tight_layout()
plt.show()

1. Outliers  
- What You Observed: Petal length and petal width have high values on the far left near zero.
- Interpretation: These might not be outliers, but rather low values in the distribution. Given that the petal width and petal length can be small, the value near zero is probably part of the natural distribution (for example, some flowers might have very small petals). However, if you notice isolated, extreme values that are distant from the main cluster, those would be actual outliers.

2. Skewness
- What You Observed: Not sure if any plot is skewed.
- Interpretation: Skewness occurs when the histogram has a long tail on one side (right or left). In your case:
- Petal width and petal length look slightly right-skewed (long tail towards larger values).
- Sepal length and sepal width seem more balanced, but sepal width could also have a slight right skew.
- How to Check: Look for long tails. A right skew means a longer tail on the right side, and left skew means a longer tail on the left.

3. Bimodal/Multimodal
- What You Observed: Petal length and petal width seem to have 2 peaks.
- Interpretation: You are correct! These features appear bimodal, meaning there are two distinct groups or clusters in the data. This could reflect the different species in the Iris dataset (e.g., Setosa might have smaller petals and Versicolor and Virginica might have larger ones).
- Why It Matters: Bimodal distributions are common when your data has distinct categories, and it’s a good indication that different species are being represented with separate characteristics.

4. Uniform Distribution
- What You Observed: No uniform distribution.
- Interpretation: Exactly, uniform distributions aren’t really present here. Uniform distributions would look like a flat histogram where each bin has roughly the same count, indicating no clear trend. None of your features exhibit this behavior, so there’s no concern here.

5. Normal Distribution
- What You Observed: Sepal width seems to have a normal shape.
- Interpretation: Sepal width does look relatively bell-shaped in the histogram, which is close to the ideal normal distribution. However, there’s some skewing towards the left. True normality would show a perfectly symmetrical bell curve, but this is still relatively close to normal.

6. Gaps
- What You Observed: Petal length and petal width have a gap.
- Interpretation: The gaps you observe could be natural and reflect a lack of data in those ranges, or it could indicate missing data or undersampling in those areas. For example, there might not be many iris flowers with very small or very large petals in the dataset.

#### No clear anomalies --> keep everything and proceed

---

### Preprocess the Data
- Normalization: Normalize the input features to have a mean of 0 and a standard deviation of 1 (use StandardScaler from sklearn.preprocessing).
- Train/Test Split: Split the data into training and testing sets. Use 80% of the data for training and 20% for testing.

In [ ]:
display(x)

In [ ]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)
# became an array --> better for the training, but I want to plot the scaled data

x_scaled_df = pd.DataFrame(x_scaled, columns=x.columns)
plt.figure(figsize=(10,10))
for i, col in enumerate(x_scaled_df):
    plot_idx = i + 1
    plt.subplot(4,4, plot_idx)
    sns.histplot(x_scaled_df[col], kde=True)
    plt.title(col)

plt.tight_layout()
plt.show()
    
# same as before but centered in mean = 0 and has variance = 1

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.2, random_state=42)

---

> # NEW STUFF
#### Define the Model

> When you are working with a classification problem (like the Iris dataset), the goal of the neural network’s output layer is to predict which class the input data belongs to.

You will define a simple feedforward neural network model in PyTorch using nn.Module. The model will consist of:  
- A layer that gets 4 features as input and returns 2.
- Followed by a ReLU activation.
- An output layer that has 2 features as inputs and returns 3, these 3 outputs has to be corresponding to the three classes (0,1,2).
    - The output layer in your network has 3 units because the Iris dataset has 3 different species, which represent 3 classes.
	- These 3 output values (logits) represent the raw predictions from the network, one for each class. For example, these could be values like [2.1, -1.5, 0.8]. These logits are not probabilities yet, they are just raw scores.
    - CrossEntropyLoss (when used for multi-class classification) will apply Softmax internally to the raw logits.
	- Softmax converts these logits into probabilities that sum to 1, assigning a probability for each of the 3 classes. For instance, the logits [2.1, -1.5, 0.8] might be transformed into probabilities like [0.70, 0.05, 0.25], meaning the model thinks class 0 has a 70% chance, class 1 has a 5% chance, and class 2 has a 25% chance.
- Followed by Softmax activation --> automatically done by the CrossEntropyLoss() loss function (DO NOT PLACE IT IN THE NETWORK)

In [ ]:
class irisNN(nn.Module):
    def __init__(self):
        super(irisNN, self).__init__()

        # define all the layers + activation functions
        self.layer_1 = nn.Linear(4,2)
        self.relu = nn.ReLU()
        self.layer_2 = nn.Linear(2,3)
        # The SoftMax doesn't go here, performed by the CrossEntropyLoss()

    def forward(self, x):
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        return x

# Create a TensorDataset and DataLoader
- TensorDataset(x_train, y_train): This creates a dataset object that pairs the input data (x_train) with the target labels (y_train). Essentially, it organizes the data in a way that makes it easy to pass into a DataLoader. Each element in the dataset consists of a pair: an input and its corresponding label.

- batch_size = 12: This sets the batch size to 12. This means that during training, 12 samples will be processed at once in each forward and backward pass. The model will update its weights after processing these 12 samples.
    - Batch size is not related to the number of features (columns) in the dataset. The batch size simply refers to how many samples are processed together in each forward/backward pass during training.

- train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True): This creates a DataLoader object that takes the dataset (train_dataset) and splits it into batches of size 12. It also shuffles the dataset before splitting it into batches (this helps with randomness and can improve generalization).

In [ ]:
# to use TensorDataset and DataLoader and the model in general I need to convert Numpy arrays into Torch tensors
x_train = torch.tensor(x_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

train_dataset = TensorDataset(x_train, y_train)
batch_size = 10
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

## Define model, loss function, and optimizer

In [ ]:
model = irisNN()
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## TRAIN THE MODEL

In [ ]:
num_epochs = 100

see_results= True
for epoch in range(num_epochs):
    for x_batch, y_batch in train_loader:
        # reset gradient
        optimizer.zero_grad()

        # compute logits passing x_batch into the whole layers of the network
        logits = model(x_batch)

        if see_results:
            # Apply Softmax to get probabilities (we can skip Softmax in loss function, as it's done internally)
            probabilities = F.softmax(logits, dim=1)  # Apply Softmax to logits

            # Get predicted class label (class with the highest probability)
            _, predictions = torch.max(probabilities, dim=1)

        # compute loss: distance between model predictions and actual labels/tagets/classes
        loss = loss_fn(logits, y_batch)

        # compute gradient, needed for the optimizer to get bettwer weigths
        loss.backward()

        # update parameters taking a step with the optimizer
        optimizer.step()
    
        # Print actual targets, predictions, and loss every 10 epochs
        if epoch % 10 == 0:
            print(f"Epoch {epoch}/{num_epochs}, Loss: {loss.item()}")
            print(f"Actual labels: {y_batch}")
            print(f"Predictions: {predictions}")
            print(f"Predicted probabilities: {probabilities}")
            print("----------")

